# M7 — Automation and reproducibility

Run the selected workflow from a single entry point and inspect its recorded evidence. Notebooks remain the instructional record; the small package contains reusable calculations. No database, API server, dashboard or broker integration is required.

Run cells in order with the project environment (`uv run jupyter lab`). Paths support the project root or `notebooks/`. See [workshop order](../_docs/workshop_order.md) for dependencies and checkpoints. Numerical outputs come from actual local inputs; no synthetic data are substituted for market results.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "config/project.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/project.yaml").exists(), "Launch from the project root or notebooks/"
from uranium_signals.config import load_config
from uranium_signals.universe import load_universe
from uranium_signals.providers import acquire
from uranium_signals.run import research_tables
cfg = load_config(ROOT / "config/project.yaml")
universe = load_universe(cfg)
tickers = set(universe.ticker) | {cfg["benchmark"], cfg["broad_benchmark"]} | set(cfg["optional_benchmarks"])


## 1. Inspect configuration and package boundaries

Configuration fixes the research universe, dates, information lag, eligibility, strategy costs and basket rule. Provider acquisition, data checks, features, targets, modeling, portfolios, long-horizon analysis and reporting are separate modules. The CLI fails with non-zero status on material errors and preserves a failure record.

In [ ]:
print((ROOT / "config/project.yaml").read_text())
print("Modules:", sorted(p.name for p in (ROOT / "src/uranium_signals").glob("*.py")))


## 2. Run from the command line

From the project root, use `uv run python -m uranium_signals.run --config config/project.yaml`. Add `--offline` to require the exact local cache, or `--refresh` to acquire a new snapshot. Do not combine them. Keep the end date explicit; edit it deliberately for a later research run. The cell below runs the same entry point with the current environment and cached data.

In [ ]:
import subprocess
import sys
completed = subprocess.run([sys.executable, "-m", "uranium_signals.run", "--config", str(ROOT / "config/project.yaml"), "--offline"], cwd=ROOT, capture_output=True, text=True)
print(completed.stdout)
print(completed.stderr[-3000:])
assert completed.returncode == 0, "CLI failed; inspect the logged error"


## 3. Verify artifacts and metadata

Each run records input checksums, configuration, dependency versions, model fit cutoffs, predictions, costs/trades, latest monthly signals, and both evaluation reports. The latest signal is the last completed month-end, not a real-time quote. Only load the local model pickle you created; external pickle files are unsafe.

In [ ]:
import hashlib
import json
run_dir = ROOT / (ROOT / "reports/latest_run.txt").read_text().strip()
metadata = json.loads((run_dir / "metadata.json").read_text())
assert metadata["status"] == "success"
for name, digest in metadata["outputs_sha256"].items():
    assert hashlib.sha256((run_dir / name).read_bytes()).hexdigest() == digest
print("Report:", run_dir / "report.html")
print("Selected model:", metadata["selected_model"])
print("Signal as of:", metadata["signal_as_of"])
display(pd.read_csv(run_dir / "latest_signals.csv"))


## 4. Scheduling and final checkpoint

A manual GitHub Actions workflow is supplied; no remote workflow or recurring schedule has been activated. Local scheduling instructions are in README. Keep provider access/terms, credentials if ever needed, and failures visible. Reports and raw inputs are not automatically published.

Run `uv run pytest -q`, then repeat the CLI offline. Confirm that financial results reproduce from the same cached inputs. Record limitations even if the model underperforms. A carefully evaluated negative result meets the research objective.